In [3]:
# dependencies installation
!pip install ultralytics opencv-python pytesseract

In [8]:
# importing dependencies
import cv2
import pytesseract
import requests
from ultralytics import YOLO

In [25]:
# Load YOLO model
model = YOLO("yolov8n.pt")

# Load image
image_path = "/content/drive/MyDrive/Colab_Notebooks/coke.jpg"
image = cv2.imread(image_path)

In [26]:
# ---------- 1. Fetch Beverage Brands from OpenFoodFacts ----------
authentic_brands = [
"coca cola", "pepsi", "sprite", "fanta", "mountain dew", "7up", "dr pepper",
"red bull", "tropicana", "lipton", "nestle", "gatorade", "monster",
"snapple", "arizona", "bai", "vitaminwater", "smartwater"
]
print(f"✔ {len(authentic_brands)} global beverage brands loaded")

✔ 18 global beverage brands loaded


In [27]:
# ---------- 2. Preprocess Image for Better OCR ----------
def preprocess_for_ocr(cropped_img):
    gray = cv2.cvtColor(cropped_img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (3, 3), 0)
    _, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    resized = cv2.resize(thresh, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    return resized

In [28]:
# ---------- 3. OCR with Better Configs ----------
custom_config = r'--oem 3 --psm 6'  # LSTM OCR engine, Assume a block of text

# ---------- 4. YOLO + OCR + Authenticity Check ----------
results = model(image_path)

for result in results:
    for box in result.boxes:
        cls = model.names[int(box.cls)]
        conf = float(box.conf)

        if cls == "bottle" and conf > 0.5:
            print(f"\nDetected bottle with confidence {conf:.2f}")

            # Crop bottle area
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cropped = image[y1:y2, x1:x2]

            # OCR with preprocessing
            processed = preprocess_for_ocr(cropped)
            text = pytesseract.image_to_string(processed, config=custom_config).lower()
            print("OCR detected text:", text.strip())

            # Check if any authentic brand appears in OCR result
            matched_brand = next((brand for brand in authentic_brands if brand in text), None)

            if matched_brand:
                print(f"✅ Authentic brand detected: {matched_brand.title()}")
            else:
                print("❌ Possibly counterfeit or unknown brand.")


image 1/1 /content/drive/MyDrive/Colab_Notebooks/coke.jpg: 640x448 1 bottle, 9.2ms
Speed: 2.3ms preprocess, 9.2ms inference, 1.6ms postprocess per image at shape (1, 3, 640, 448)

Detected bottle with confidence 0.94
OCR detected text: ;
❌ Possibly counterfeit or unknown brand.
